In [6]:
import pandas as pd
import ast
from tqdm import tqdm
import csv

def find_common_cuis(captions_file, outputs_file, output_file):
    """
    Finds common CUI_IDs between two CSV files and creates a new CSV with merged data,
    including the specific CUI that matched.

    Args:
        captions_file (str): Path to the generated captions CSV file.
        outputs_file (str): Path to the output with CUIs CSV file.
        output_file (str): Path for the output CSV file.
    """
    try:
        print("Step 1/5: Reading CSV files...")
        # Read the CSV files into pandas DataFrames
        df_captions = pd.read_csv(captions_file)
        df_outputs = pd.read_csv(outputs_file)
        print("...CSVs read successfully.")

        print("\nStep 2/5: Cleaning and Preparing Data...")
        # --- Data Cleaning and Preparation ---

        # Drop the unnecessary 'Unnamed: 1' column if it exists in the outputs file
        if 'Unnamed: 1' in df_outputs.columns:
            df_outputs = df_outputs.drop(columns=['Unnamed: 1'])
            print("- Dropped 'Unnamed: 1' column from outputs file.")

        # Rename 'Generated_Caption' to 'generated_caption' for consistency
        if 'Generated_Caption' in df_captions.columns:
            df_captions = df_captions.rename(columns={'Generated_Caption': 'generated_caption'})
            print("- Renamed 'Generated_Caption' column.")

        # Function to safely parse the string representation of a list
        def parse_cui_list(cui_string):
            if pd.isna(cui_string):
                return []
            try:
                # ast.literal_eval safely evaluates a string containing a Python literal
                return ast.literal_eval(cui_string)
            except (ValueError, SyntaxError):
                # Return an empty list if the string is not a valid list format
                return []
        
        # Initialize tqdm for pandas integration
        tqdm.pandas(desc="Parsing Captions CUIs")
        # Convert the 'CUI_IDs' string column into a list of strings for both dataframes
        df_captions['CUI_List'] = df_captions['CUI_IDs'].progress_apply(parse_cui_list)
        
        tqdm.pandas(desc="Parsing Outputs CUIs")
        df_outputs['CUI_List'] = df_outputs['CUI_IDs'].progress_apply(parse_cui_list)
        print("...Data preparation complete.")


        print("\nStep 3/5: Matching Logic...")
        # --- Matching Logic ---

        print("- Exploding dataframes for matching...")
        # Create a new row for each CUI in the list (exploding the list)
        # This makes it easier to find matches
        captions_exploded = df_captions.explode('CUI_List')
        outputs_exploded = df_outputs.explode('CUI_List')

        print("- Merging dataframes on CUI_List...")
        # Merge the two dataframes based on the common CUI values
        # This will create a row for every single CUI match found
        merged_df = pd.merge(
            captions_exploded,
            outputs_exploded,
            left_on='CUI_List',
            right_on='CUI_List',
            suffixes=('_caption', '_output')
        )
        print("...Matching complete.")


        print("\nStep 4/5: Final Output Formatting...")
        # --- Final Output Formatting ---

        # Select the desired columns for the final output, including the matching CUI
        result_df = merged_df[['ROCO_ID', 'generated_caption', 'reference_summary', 'CUI_List']]
        
        # Rename the 'CUI_List' column to be more descriptive
        result_df = result_df.rename(columns={'CUI_List': 'Matching_CUI_ID'})
        
        # Keeping duplicate rows as requested
        result_df = result_df.reset_index(drop=True)
        print("- Final columns selected: ROCO_ID, generated_caption, reference_summary, Matching_CUI_ID")

        print("\nStep 5/5: Saving the final result...")
        # Save the final result to a new CSV file, quoting all fields to handle special characters.
        result_df.to_csv(output_file, index=False, quoting=csv.QUOTE_ALL)
        print(f"\nSuccessfully created '{output_file}' with {len(result_df)} matched records.")

    except FileNotFoundError as e:
        print(f"Error: {e}. Please make sure the input files are in the same directory as the script.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


# --- Main execution block ---
if __name__ == "__main__":
    # Define the names of your input and output files
# Define the names of your input and output files
    captions_csv = "C:\\.Final_Year_Project\\project\\generated_captions_updated.csv"
    outputs_csv = "C:\\.Final_Year_Project\\project\\output_with_CUIs.csv"
    output_csv = "C:\\.Final_Year_Project\\project\\matched_roco_summaries.csv"


    # Run the function
    find_common_cuis(captions_csv, outputs_csv, output_csv)

Step 1/5: Reading CSV files...
...CSVs read successfully.

Step 2/5: Cleaning and Preparing Data...
- Dropped 'Unnamed: 1' column from outputs file.
- Renamed 'Generated_Caption' column.


Parsing Outputs CUIs: 100%|██████████| 1301/1301 [00:00<00:00, 21292.96it/s]

...Data preparation complete.

Step 3/5: Matching Logic...
- Exploding dataframes for matching...
- Merging dataframes on CUI_List...
...Matching complete.

Step 4/5: Final Output Formatting...
- Final columns selected: ROCO_ID, generated_caption, reference_summary, Matching_CUI_ID

Step 5/5: Saving the final result...

Successfully created 'C:\.Final_Year_Project\project\matched_roco_summaries.csv' with 1 matched records.
